# AcousticAcre: train both models
1. **Runtime → Change runtime type → T4 GPU** (needed for the crop model).
2. Upload `acousticacre.zip` with the folder icon on the left.
3. Run the cells in order. Total time about 45–75 min, mostly the crop training.

Report only what the scripts print. Machine-sound results come from **factory pumps (MIMII)**, cassava from **field photos**, maize/tomato from **lab photos**. Say which is which.

In [ ]:
!unzip -q -o acousticacre.zip
!pip -q install timm onnx onnxruntime importlib_resources
import torch; print('GPU:', torch.cuda.is_available())

## 1. Machine sounds (real industrial pumps, MIMII via DCASE 2020)
If `ls` shows a tiny file, open zenodo.org/records/3678171, right-click the pump file, copy its link and paste it below.

In [ ]:
!wget -q -O pump.zip "https://zenodo.org/records/3678171/files/dev_data_pump.zip?download=1"
!ls -lh pump.zip

In [ ]:
!unzip -q -o pump.zip -d dcase
!python acousticacre/tools/import_dcase.py dcase acousticacre/ml/data/raw --per-class 150 --copy
%cd /content/acousticacre/ml
!python -m train.audio_baseline --data data/raw --out runs/baseline
!python -m export.baseline_to_onnx --run runs/baseline
!python -c "from dsp.logmel import export_constants; export_constants('runs/baseline/dsp_constants.json')"
%cd /content/acousticacre
!python tools/pack_builder.py machine --run ml/runs/baseline --out packs

## 2. Crop photos (cassava field photos + PlantVillage maize/tomato lab photos)
Downloads about 2 GB through TensorFlow Datasets. Training: about 30–60 min on a T4.

If the PlantVillage download fails or hangs, rerun the first line with `--skip-plantvillage` (cassava only).

In [ ]:
%cd /content/acousticacre/ml
!python -m train.crop_prepare --out data/crop --max-per-class 1500
!python -m train.crop_train --data data/crop --out runs/crop --epochs 8 --quantize
%cd /content/acousticacre
!python tools/pack_builder.py crop --run ml/runs/crop --out packs

## 3. Download results
Send the two `report.json` files back for the results slide; give the `.aapack` files to the frontend dev.

In [ ]:
import shutil
from google.colab import files
shutil.copy('/content/acousticacre/ml/runs/baseline/report.json', '/content/machine_report.json')
shutil.copy('/content/acousticacre/ml/runs/crop/report.json', '/content/crop_report.json')
for f in ['/content/machine_report.json', '/content/crop_report.json',
          '/content/acousticacre/packs/machine-core-1-0.1.0.aapack', '/content/acousticacre/packs/crop-core-1-0.1.0.aapack']:
    files.download(f)